# ASAS-SN per-camera masked-GP audit for reviewed microlensing candidates

This notebook evaluates `per_camera_gp_baseline_masked` on the **deduplicated union** of candidates marked either:

- `reviews.event_class = 'microlensing'`, or
- `possible_microlensing_event` in the scalar or JSON secondary morphology tags.

It reads the live July 1 Review database in SQLite read-only mode, resolves each bundled ASAS-SN light curve through the same Review path resolver used by MALCA, applies canonical row cleaning, and then calls the production baseline function with the STV default baseline arguments. It does **not** remove cameras or fit a microlensing model.

The plots are intended to answer a specific question: does the masked GP follow the quiescent camera baseline while leaving a candidate brightening event in the residuals? Screening flags below prioritize visual review; they are not scientific pass/fail criteria.

In [ ]:
from __future__ import annotations

import json
import sqlite3
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", message=".*Covariance of the parameters could not be estimated.*")

candidate_roots = [Path.cwd(), *Path.cwd().parents]
repo_root = next(
    (root for root in candidate_roots if (root / "malca" / "core" / "baseline.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("Run this notebook from inside the MALCA checkout.")
repo_root = repo_root.resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from malca.config import GP_BRIGHT_SIGMA_THRESH, GP_DIP_SIGMA_THRESH
from malca.core.baseline import per_camera_gp_baseline_masked
from malca.core.utils import clean_lc
from malca.io.lightcurve_io import (
    load_lightcurve_df,
    stable_camera_color,
    to_asassn_algorithm_frame,
)
from malca.review.native_lightcurve import resolve_lightcurve_path
from malca.review.store import get_candidate_payload
from malca.stv.events import DEFAULT_BASELINE_KWARGS

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
print(f"Repository: {repo_root}")

In [ ]:
# Live Review source. Change this path only if you deliberately want to audit another run.
RUN_ROOT = repo_root / "output" / "runs" / "dat3-full-extended_2026-07-01-v4"
REVIEW_DB = RUN_ROOT / "review" / "review.db"

# The notebook always evaluates the complete primary-or-possible microlensing union.
BASELINE_KWARGS = dict(DEFAULT_BASELINE_KWARGS)

# These thresholds only sort candidates for closer inspection; they do not reject anything.
QUIET_SCATTER_ALERT_MAG = 0.10
QUIET_MEAN_SIGMA2_ALERT = 4.0

# Optional exports are separate from the Review DB and never modify it.
WRITE_SUMMARY_CSV = False
SUMMARY_DIR = RUN_ROOT / "results" / "microlensing_gp_baseline_audit"

if not REVIEW_DB.is_file():
    raise FileNotFoundError(REVIEW_DB)

display(pd.Series(BASELINE_KWARGS, name="production STV baseline value").to_frame())
print(f"Review DB (read-only): {REVIEW_DB}")

## Load the complete reviewed cohort

The SQL handles both the legacy scalar secondary tag and the current JSON list. `SELECT DISTINCT` prevents candidates carrying both labels from appearing twice. `label_source` is retained only as provenance; all categories are evaluated together.

In [ ]:
COHORT_SQL = r"""
WITH labeled AS (
    SELECT
        c.candidate_id,
        c.asas_sn_id,
        c.ra,
        c.dec,
        c.lc_path,
        c.source_path,
        c.jump_best_t0,
        r.event_class,
        r.morphology_secondary,
        r.morphology_secondary_json,
        lower(trim(coalesce(r.event_class, ''))) = 'microlensing' AS is_primary_microlensing,
        (
            lower(trim(coalesce(r.morphology_secondary, ''))) = 'possible_microlensing_event'
            OR EXISTS (
                SELECT 1
                FROM json_each(
                    CASE
                        WHEN json_valid(r.morphology_secondary_json)
                        THEN r.morphology_secondary_json
                        ELSE '[]'
                    END
                ) AS secondary
                WHERE lower(trim(cast(secondary.value AS TEXT))) = 'possible_microlensing_event'
            )
        ) AS is_possible_microlensing
    FROM candidates AS c
    INNER JOIN reviews AS r ON r.candidate_id = c.candidate_id
)
SELECT DISTINCT
    *,
    CASE
        WHEN is_primary_microlensing AND is_possible_microlensing THEN 'primary_and_possible'
        WHEN is_primary_microlensing THEN 'primary_only'
        ELSE 'possible_only'
    END AS label_source
FROM labeled
WHERE is_primary_microlensing OR is_possible_microlensing
ORDER BY candidate_id
"""

read_only_uri = f"file:{REVIEW_DB.as_posix()}?mode=ro"
with sqlite3.connect(read_only_uri, uri=True) as connection:
    candidates = pd.read_sql_query(COHORT_SQL, connection)
    resolved_paths = []
    for candidate_id in candidates["candidate_id"].astype(str):
        payload = get_candidate_payload(connection, candidate_id)
        resolved = resolve_lightcurve_path(payload, RUN_ROOT)
        resolved_paths.append(str(resolved.resolve()) if resolved is not None else None)

candidates["resolved_lightcurve_path"] = resolved_paths
candidates["lightcurve_exists"] = candidates["resolved_lightcurve_path"].map(
    lambda value: bool(value) and Path(value).is_file()
)

if candidates["candidate_id"].duplicated().any():
    raise AssertionError("Cohort query returned duplicate candidate IDs.")

display(candidates["label_source"].value_counts().rename_axis("label_source").to_frame("candidates"))
print(f"Union cohort: {len(candidates)} candidates")
print(f"Resolved light curves: {int(candidates['lightcurve_exists'].sum())}/{len(candidates)}")
display(candidates[["candidate_id", "asas_sn_id", "label_source", "resolved_lightcurve_path"]])

## Run the production masked-GP baseline

Each light curve is loaded through the canonical ASAS-SN adapter, converted to the event-algorithm column names, and passed through `clean_lc` before baseline estimation. Camera removal is intentionally disabled: this audit is meant to reveal camera-specific successes and failures rather than hide them.

In [ ]:
MAD_SCALE = 1.4826
BAND_NAMES = {0.0: "g", 1.0: "V"}

def robust_scatter(values) -> float:
    array = pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(float)
    array = array[np.isfinite(array)]
    if array.size == 0:
        return np.nan
    median = float(np.median(array))
    return float(MAD_SCALE * np.median(np.abs(array - median)))


def finite_percentile_span(values, low=5.0, high=95.0) -> float:
    array = pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(float)
    array = array[np.isfinite(array)]
    if array.size == 0:
        return np.nan
    return float(np.percentile(array, high) - np.percentile(array, low))


def prepare_candidate(candidate_row: pd.Series) -> tuple[pd.DataFrame, float]:
    path = Path(str(candidate_row["resolved_lightcurve_path"]))
    canonical = load_lightcurve_df(path, apply_quality=True)
    algorithm_frame = to_asassn_algorithm_frame(canonical)
    cleaned = clean_lc(algorithm_frame).reset_index(drop=True)
    if cleaned.empty:
        raise ValueError("No observations remain after canonical cleaning.")

    started = time.perf_counter()
    prepared = per_camera_gp_baseline_masked(cleaned, **BASELINE_KWARGS).reset_index(drop=True)
    elapsed = time.perf_counter() - started

    if len(prepared) != len(cleaned):
        raise AssertionError("Baseline function changed the number of rows.")
    prepared["candidate_id"] = str(candidate_row["candidate_id"])
    prepared["label_source"] = str(candidate_row["label_source"])
    prepared["band"] = pd.to_numeric(prepared["v_g_band"], errors="coerce").map(BAND_NAMES).fillna("unknown")
    prepared["camera_label"] = prepared["camera#"].astype(str)
    prepared["is_masked"] = prepared["is_masked"].fillna(False).astype(bool)
    return prepared, elapsed


def summarize_camera(candidate_row: pd.Series, camera_frame: pd.DataFrame, runtime_s: float) -> dict:
    finite = (
        np.isfinite(pd.to_numeric(camera_frame["resid"], errors="coerce"))
        & np.isfinite(pd.to_numeric(camera_frame["sigma_eff"], errors="coerce"))
        & (pd.to_numeric(camera_frame["sigma_eff"], errors="coerce") > 0)
    )
    quiet = finite & ~camera_frame["is_masked"]
    masked = finite & camera_frame["is_masked"]
    quiet_sigma = pd.to_numeric(camera_frame.loc[quiet, "sigma_resid"], errors="coerce").to_numpy(float)
    quiet_sigma = quiet_sigma[np.isfinite(quiet_sigma)]
    sources = sorted(camera_frame["baseline_source"].dropna().astype(str).unique())
    errors = pd.to_numeric(camera_frame["error"], errors="coerce").to_numpy(float)
    sigma_eff = pd.to_numeric(camera_frame["sigma_eff"], errors="coerce").to_numpy(float)
    valid_ratio = np.isfinite(errors) & (errors > 0) & np.isfinite(sigma_eff)
    return {
        "candidate_id": str(candidate_row["candidate_id"]),
        "asas_sn_id": candidate_row.get("asas_sn_id"),
        "label_source": str(candidate_row["label_source"]),
        "band": str(camera_frame["band"].iloc[0]),
        "camera": str(camera_frame["camera_label"].iloc[0]),
        "n_points": int(len(camera_frame)),
        "n_masked": int(camera_frame["is_masked"].sum()),
        "masked_fraction": float(camera_frame["is_masked"].mean()),
        "jd_span_days": finite_percentile_span(camera_frame["JD"], 0, 100),
        "quiet_residual_median_mag": float(pd.to_numeric(camera_frame.loc[quiet, "resid"], errors="coerce").median()),
        "quiet_residual_scatter_mag": robust_scatter(camera_frame.loc[quiet, "resid"]),
        "quiet_mean_sigma_resid_sq": float(np.mean(quiet_sigma**2)) if quiet_sigma.size else np.nan,
        "min_sigma_resid": float(pd.to_numeric(camera_frame.loc[finite, "sigma_resid"], errors="coerce").min()),
        "max_sigma_resid": float(pd.to_numeric(camera_frame.loc[finite, "sigma_resid"], errors="coerce").max()),
        "masked_min_sigma_resid": float(pd.to_numeric(camera_frame.loc[masked, "sigma_resid"], errors="coerce").min()) if masked.any() else np.nan,
        "baseline_p95_p05_mag": finite_percentile_span(camera_frame["baseline"]),
        "rough_to_final_scatter_mag": robust_scatter(
            pd.to_numeric(camera_frame["baseline"], errors="coerce")
            - pd.to_numeric(camera_frame["base_rough"], errors="coerce")
        ),
        "median_sigma_eff_over_error": float(np.median(sigma_eff[valid_ratio] / errors[valid_ratio])) if valid_ratio.any() else np.nan,
        "baseline_source": ",".join(sources),
        "needs_consensus": bool(camera_frame["needs_consensus"].fillna(False).astype(bool).any()),
        "cross_band_calibrated": bool(camera_frame["cross_band_calibrated"].fillna(False).astype(bool).any()),
        "candidate_runtime_s": float(runtime_s),
    }


baseline_frames: dict[str, pd.DataFrame] = {}
camera_records: list[dict] = []
candidate_records: list[dict] = []
failure_records: list[dict] = []

for position, (_, candidate_row) in enumerate(candidates.iterrows(), start=1):
    candidate_id = str(candidate_row["candidate_id"])
    print(f"[{position:02d}/{len(candidates):02d}] {candidate_id}", end=" ... ")
    if not bool(candidate_row["lightcurve_exists"]):
        failure_records.append({"candidate_id": candidate_id, "error": "lightcurve_not_found"})
        print("missing")
        continue
    try:
        prepared, runtime_s = prepare_candidate(candidate_row)
        baseline_frames[candidate_id] = prepared
        rows = [
            summarize_camera(candidate_row, camera_frame, runtime_s)
            for _, camera_frame in prepared.groupby(["band", "camera_label"], sort=True)
        ]
        camera_records.extend(rows)
        camera_table = pd.DataFrame(rows)
        flags = []
        if int(camera_table["n_masked"].sum()) == 0:
            flags.append("no_masked_points")
        if (camera_table["quiet_residual_scatter_mag"] > QUIET_SCATTER_ALERT_MAG).any():
            flags.append("large_quiet_scatter")
        if (camera_table["quiet_mean_sigma_resid_sq"] > QUIET_MEAN_SIGMA2_ALERT).any():
            flags.append("large_standardized_scatter")
        if camera_table["needs_consensus"].any():
            flags.append("consensus_used")
        required = ["baseline", "resid", "sigma_eff"]
        nonfinite_required = sum(int(~np.isfinite(pd.to_numeric(prepared[col], errors="coerce")).all()) for col in required)
        if nonfinite_required:
            flags.append("nonfinite_output")
        candidate_records.append({
            "candidate_id": candidate_id,
            "asas_sn_id": candidate_row.get("asas_sn_id"),
            "label_source": str(candidate_row["label_source"]),
            "n_points": int(len(prepared)),
            "n_cameras": int(prepared["camera_label"].nunique()),
            "n_band_camera_groups": int(len(camera_table)),
            "n_masked": int(prepared["is_masked"].sum()),
            "masked_fraction": float(prepared["is_masked"].mean()),
            "max_quiet_residual_scatter_mag": float(camera_table["quiet_residual_scatter_mag"].max()),
            "max_quiet_mean_sigma_resid_sq": float(camera_table["quiet_mean_sigma_resid_sq"].max()),
            "max_baseline_p95_p05_mag": float(camera_table["baseline_p95_p05_mag"].max()),
            "baseline_sources": ",".join(sorted(set(camera_table["baseline_source"]))),
            "runtime_s": float(runtime_s),
            "screening_flags": ",".join(flags) if flags else "none",
        })
        print(f"{len(prepared)} points, {len(camera_table)} band-camera groups, {runtime_s:.3f} s")
    except Exception as exc:
        failure_records.append({"candidate_id": candidate_id, "error": f"{type(exc).__name__}: {exc}"})
        print(f"FAILED: {type(exc).__name__}: {exc}")

candidate_summary = pd.DataFrame(candidate_records).sort_values(
    ["screening_flags", "max_quiet_mean_sigma_resid_sq"], ascending=[True, False]
).reset_index(drop=True)
camera_summary = pd.DataFrame(camera_records).sort_values(
    ["quiet_mean_sigma_resid_sq", "quiet_residual_scatter_mag"], ascending=False
).reset_index(drop=True)
failures = pd.DataFrame(failure_records, columns=["candidate_id", "error"])

print(f"\nSuccessful: {len(candidate_summary)}/{len(candidates)}; failures: {len(failures)}")

## Population summary

Useful interpretations:

- `masked_fraction`: how much of the light curve the first pass considered an excursion and withheld from the final GP.
- `quiet_residual_scatter_mag`: robust residual scatter outside the mask.
- `quiet_mean_sigma_resid_sq`: mean squared standardized residual outside the mask; values far above one indicate remaining dispersion relative to `sigma_eff`.
- `baseline_p95_p05_mag`: robust amplitude of the final time-varying baseline. Large values require visual inspection because they can represent real baseline drift or excessive flexibility.
- `rough_to_final_scatter_mag`: how substantially the flexible final solution differs from the initial stiff baseline.

None of these metrics alone establishes that the GP preserved or absorbed a microlensing event. The detailed plots below are the decisive audit.

In [ ]:
display(candidate_summary)
if not failures.empty:
    display(Markdown("### Failures"))
    display(failures)

if not candidate_summary.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes = axes.ravel()
    label_colors = {
        "primary_and_possible": "#6f4e9c",
        "primary_only": "#1f77b4",
        "possible_only": "#dd8452",
    }
    for label_source, sub in candidate_summary.groupby("label_source"):
        color = label_colors.get(label_source, "0.4")
        axes[0].scatter(sub["n_points"], sub["runtime_s"], s=35, alpha=0.8, color=color, label=label_source)
        axes[1].scatter(sub["masked_fraction"], sub["max_quiet_residual_scatter_mag"], s=35, alpha=0.8, color=color)
        axes[2].scatter(sub["max_baseline_p95_p05_mag"], sub["max_quiet_mean_sigma_resid_sq"], s=35, alpha=0.8, color=color)
    axes[0].set(xlabel="cleaned observations", ylabel="baseline runtime (s)", title="Computational runtime")
    axes[0].legend(fontsize=8)
    axes[1].axhline(QUIET_SCATTER_ALERT_MAG, color="crimson", linestyle="--", linewidth=1)
    axes[1].set(xlabel="masked fraction", ylabel="maximum camera quiet scatter (mag)", title="Masking and residual scatter")
    axes[2].axhline(QUIET_MEAN_SIGMA2_ALERT, color="crimson", linestyle="--", linewidth=1)
    axes[2].set(xlabel="maximum camera baseline p95-p05 (mag)", ylabel="maximum quiet mean $\sigma_{resid}^2$", title="Baseline flexibility and standardized scatter")
    candidate_summary["screening_flags"].value_counts().sort_values().plot.barh(ax=axes[3], color="#4c72b0")
    axes[3].set(xlabel="candidates", ylabel="screening flags", title="Visual-review prioritization")
    fig.tight_layout()
    plt.show()

## Detailed baseline atlas

Each candidate has one column per available passband:

1. Cleaned magnitude measurements, initial stiff `base_rough` (dotted), and final masked-GP `baseline` (solid). Masked points are marked with crosses.
2. Magnitude residuals (`mag - baseline`). A microlensing brightening remains numerically negative rather than being followed by the final baseline; the display axis is inverted so brighter excursions appear upward.
3. Standardized residuals, also displayed with the magnitude-sign axis inverted. The horizontal dashed lines retain their numerical configured masking thresholds.

Camera colors are stable across candidates. The vertical gray line is the stored Review jump epoch when available; it is contextual metadata, not an input to the GP.

In [ ]:
REDUCED_JD_OFFSET = 2450000.0

def _reduced_jd(values):
    array = pd.to_numeric(values, errors="coerce").to_numpy(float)
    finite = array[np.isfinite(array)]
    if finite.size and float(np.nanmedian(finite)) > 1_000_000:
        return array - REDUCED_JD_OFFSET
    return array


def plot_candidate_baseline(candidate_row: pd.Series, prepared: pd.DataFrame):
    ordered_bands = [band for band in ("V", "g", "unknown") if band in set(prepared["band"])]
    ncols = max(len(ordered_bands), 1)
    fig, axes = plt.subplots(3, ncols, figsize=(8.2 * ncols, 10.5), sharex="col", squeeze=False)

    jump_t0 = pd.to_numeric(pd.Series([candidate_row.get("jump_best_t0")]), errors="coerce").iloc[0]
    if np.isfinite(jump_t0) and jump_t0 > 1_000_000:
        jump_t0 -= REDUCED_JD_OFFSET

    for column, band in enumerate(ordered_bands):
        band_frame = prepared.loc[prepared["band"] == band]
        for camera, camera_frame in band_frame.groupby("camera_label", sort=True):
            camera_frame = camera_frame.sort_values("JD")
            x = _reduced_jd(camera_frame["JD"])
            color = stable_camera_color(camera)
            masked = camera_frame["is_masked"].to_numpy(bool)
            quiet = ~masked

            axes[0, column].errorbar(
                x[quiet], camera_frame.loc[quiet, "mag"], yerr=camera_frame.loc[quiet, "error"],
                fmt=".", markersize=3, linewidth=0.4, alpha=0.45, color=color, label=f"camera {camera}",
            )
            if masked.any():
                axes[0, column].scatter(
                    x[masked], camera_frame.loc[masked, "mag"], marker="x", s=28, linewidth=1.0, color=color,
                )
            axes[0, column].plot(x, camera_frame["base_rough"], linestyle=":", linewidth=1.0, alpha=0.7, color=color)
            axes[0, column].plot(x, camera_frame["baseline"], linestyle="-", linewidth=1.7, color=color)

            axes[1, column].scatter(x[quiet], camera_frame.loc[quiet, "resid"], s=10, alpha=0.45, color=color)
            if masked.any():
                axes[1, column].scatter(x[masked], camera_frame.loc[masked, "resid"], marker="x", s=28, linewidth=1.0, color=color)
            axes[2, column].scatter(x[quiet], camera_frame.loc[quiet, "sigma_resid"], s=10, alpha=0.45, color=color)
            if masked.any():
                axes[2, column].scatter(x[masked], camera_frame.loc[masked, "sigma_resid"], marker="x", s=28, linewidth=1.0, color=color)

        axes[0, column].invert_yaxis()
        axes[0, column].set_title(f"{band} band: data, rough baseline, final baseline")
        axes[0, column].set_ylabel("magnitude")
        axes[0, column].legend(fontsize=8, ncol=2)
        axes[1, column].axhline(0, color="0.2", linewidth=0.8)
        axes[1, column].invert_yaxis()
        axes[1, column].set_ylabel("mag - baseline")
        axes[2, column].axhline(0, color="0.2", linewidth=0.8)
        axes[2, column].axhline(GP_BRIGHT_SIGMA_THRESH, color="#c44e52", linestyle="--", linewidth=1, label="bright mask threshold")
        axes[2, column].axhline(GP_DIP_SIGMA_THRESH, color="#4c72b0", linestyle="--", linewidth=1, label="dip mask threshold")
        axes[2, column].invert_yaxis()
        axes[2, column].set_ylabel("standardized residual")
        axes[2, column].set_xlabel("JD - 2,450,000")
        axes[2, column].legend(fontsize=8)
        if np.isfinite(jump_t0):
            for row in range(3):
                axes[row, column].axvline(jump_t0, color="0.35", linestyle="--", linewidth=0.9, alpha=0.8)

    candidate_id = str(candidate_row["candidate_id"])
    summary_row = candidate_summary.loc[candidate_summary["candidate_id"] == candidate_id].iloc[0]
    fig.suptitle(
        f"{candidate_id} | ASAS-SN {candidate_row.get('asas_sn_id')} | {candidate_row['label_source']}\n"
        f"{summary_row['n_points']} points, {summary_row['n_band_camera_groups']} band-camera groups, "
        f"masked={summary_row['masked_fraction']:.3f}, flags={summary_row['screening_flags']}",
        fontsize=14,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    return fig


for _, candidate_row in candidates.iterrows():
    candidate_id = str(candidate_row["candidate_id"])
    prepared = baseline_frames.get(candidate_id)
    if prepared is None:
        continue
    display(Markdown(f"### `{candidate_id}`"))
    figure = plot_candidate_baseline(candidate_row, prepared)
    plt.show()
    plt.close(figure)

## Camera-level triage and optional exports

The first table is deliberately sorted by standardized quiet scatter, followed by magnitude scatter. Inspect its highest rows against the candidate atlas. If desired, set `WRITE_SUMMARY_CSV = True` near the top and rerun this cell to write non-destructive CSV summaries under the run's `results/` directory.

In [ ]:
display(camera_summary)

if WRITE_SUMMARY_CSV:
    SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
    candidate_path = SUMMARY_DIR / "candidate_summary.csv"
    camera_path = SUMMARY_DIR / "camera_summary.csv"
    failure_path = SUMMARY_DIR / "failures.csv"
    candidate_summary.to_csv(candidate_path, index=False)
    camera_summary.to_csv(camera_path, index=False)
    failures.to_csv(failure_path, index=False)
    print(candidate_path)
    print(camera_path)
    print(failure_path)
else:
    print("WRITE_SUMMARY_CSV is False; no files were written.")

## What to look for before adopting this baseline in microlensing fitting

For each candidate, check:

1. **Event preservation:** the final solid baseline should pass through the quiescent measurements, not the brightening excursion. The event should remain as a coherent negative residual.
2. **Mask completeness:** the crosses should cover the full event, including its wings. Unmasked event wings can pull the flexible second-pass GP toward the event.
3. **Camera transitions:** residual medians should not jump systematically when the active camera changes.
4. **Long-timescale risk:** candidates whose event occupies much of a camera's observing span are the most likely to lack a trustworthy quiescent reference. Pay special attention to `consensus_used` cases.
5. **Uncertainty realism:** quiet standardized residuals should be broadly centered on zero without extreme overdispersion.
6. **Band behavior:** inspect `g` and `V` separately. The masked GP may perform differently when their temporal coverage differs.

Even if these plots look good, use the GP only to define quiescent reference levels and effective uncertainties for microlensing. Do not fit the PSPL model to GP residuals or subtract the full time-varying GP baseline without a separate injection-recovery demonstration.